# TMFT End-to-End Experiment

Run this notebook from the uploaded `tmft_project/` directory. It prepares real PII-containing Enron splits, trains all five methods, and computes TER, SER, perplexity, MDP, and two MIA AUC metrics.

## 0. Verify Project Structure

In [ ]:
from pathlib import Path
import os, sys, platform

PROJECT_ROOT = Path.cwd()
required = [
    'configs/config.yaml', 'src/data_prep.py', 'src/train.py',
    'src/masking.py', 'src/evaluate_pii.py', 'src/evaluate_ppl.py',
    'src/evaluate_mia.py', 'src/plot_results.py', 'main.py', 'requirements.txt'
]
missing = [path for path in required if not (PROJECT_ROOT / path).exists()]
if missing:
    raise FileNotFoundError('Open the notebook inside the full tmft_project directory. Missing: ' + ', '.join(missing))
print('Project:', PROJECT_ROOT)
print('Python:', sys.version)
print('Platform:', platform.platform())

## 1. Install Dependencies
Run once in a fresh Vessel workspace, then restart the kernel.

In [ ]:
%pip install -U pip
%pip uninstall -y transformers peft accelerate tokenizers huggingface_hub datasets
%pip install -r requirements.txt
!python -m spacy download en_core_web_sm
print('Restart the kernel now, then continue from the next cell.')

## 2. Imports and GPU Check

In [ ]:
import os
os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')

import pandas as pd
import torch
from src.data_prep import prepare_experiment_data
from src.plot_results import plot_results
from src.train import METHODS, load_config, train_model, upload_to_huggingface
from main import run_eval

print('Methods:', METHODS)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Smoke-Test Data Preparation

In [ ]:
smoke_config = load_config('configs/config.yaml')
smoke_config.update({
    'max_train_samples': 100,
    'max_prepared_samples': 80,
    'max_eval_samples': 20,
    'prepared_data_dir': './data/smoke_processed',
    'pii_eval_path': './data/smoke_pii_eval.json',
    'num_epochs': 0.02,
    'batch_size': 1,
    'gradient_accumulation_steps': 1,
    'fp16': False,
    'output_dir': './results/smoke',
})
smoke_splits, smoke_eval_path = prepare_experiment_data(smoke_config, force=True)
print(smoke_splits)
print('Real evaluation records:', smoke_eval_path)

## 4. Smoke-Test Training

In [ ]:
trainer, tokenizer, smoke_output = train_model(
    smoke_config,
    method='tmft_ner',
    train_dataset=smoke_splits['train'],
    eval_dataset=smoke_splits['validation'],
)
print('Saved:', smoke_output)

## 5. Final Data Preparation
This rebuilds real PII-containing train/validation/test splits and automatically creates `data/pii_eval.json`.

In [ ]:
final_config = load_config('configs/config.yaml')
final_config['fp16'] = False
final_splits, eval_path = prepare_experiment_data(final_config, force=True)
final_config['text_column'] = 'text'
print(final_splits)
print('PII eval file:', eval_path)
print('Train sample:', final_splits['train'][0]['text'][:500])

## 6. Train Five Conditions
For a time-limited pilot, use the first three methods. The final report should use all five.

In [ ]:
methods_to_run = ['baseline', 'rmft', 'tmft_ner', 'tmft_mia', 'tmft_combined']
train_outputs = {}
for method in methods_to_run:
    print(f'\n===== TRAIN: {method} =====')
    trainer, tokenizer, output_dir = train_model(
        final_config,
        method=method,
        train_dataset=final_splits['train'],
        eval_dataset=final_splits['validation'],
    )
    train_outputs[method] = str(output_dir)
train_outputs

## 7. Evaluate Privacy and Utility
Computes TER, SER, held-out PPL/MDP, Loss-MIA AUC, and Min-K MIA AUC.

In [ ]:
results_df = run_eval(final_config, 'all', final_splits, eval_path)
results_df

## 8. Generate Submission Figures

In [ ]:
csv_path = Path(final_config['results_table_dir']) / 'main_results.csv'
figure_paths = plot_results(csv_path, 'results/figures')
print('Table:', csv_path)
print('Figures:', figure_paths)

## 9. Additional Controlled Ablation Experiments

These runs reuse the existing prepared split and evaluation set. Baseline perplexity is loaded dynamically from `results/tables/main_results.csv`; if that table is missing, the saved baseline checkpoint is evaluated once. Existing ablation adapters are reused so an interrupted run can resume without retraining completed conditions.

In [ ]:
from copy import deepcopy
import gc
from pathlib import Path

import pandas as pd
import torch
from datasets import load_from_disk

from src.train import load_config, train_model
from main import run_eval

# Restore state safely after a kernel restart.
if 'final_config' not in globals():
    final_config = load_config('configs/config.yaml')
if 'final_splits' not in globals():
    final_splits = load_from_disk(final_config.get('prepared_data_dir', 'data/processed'))
if 'eval_path' not in globals():
    eval_path = Path(final_config.get('pii_eval_path', 'data/pii_eval.json'))

final_config['text_column'] = 'text'
main_results_path = Path('results/tables/main_results.csv')

# Dynamic baseline: read the original result table when available.
if main_results_path.exists():
    original_results = pd.read_csv(main_results_path)
    baseline_rows = original_results.loc[original_results['method'] == 'baseline', 'ppl']
    if baseline_rows.empty:
        raise ValueError(f'Baseline row is missing from {main_results_path}')
    baseline_ppl = float(baseline_rows.iloc[0])
    print(f'Baseline PPL loaded from table: {baseline_ppl:.6f}')
else:
    # Re-evaluate the saved baseline with the same validation split.
    baseline_cfg = deepcopy(final_config)
    baseline_cfg['results_table_dir'] = './results/ablations/baseline_reference/tables'
    baseline_model_dir = Path(final_config.get('output_dir', './results')) / 'baseline'
    if not baseline_model_dir.exists():
        raise FileNotFoundError(f'Baseline checkpoint not found: {baseline_model_dir}')
    baseline_result = run_eval(
        baseline_cfg,
        method='baseline',
        splits=final_splits,
        eval_path=eval_path,
        model_dir=str(baseline_model_dir),
    )
    baseline_ppl = float(baseline_result['ppl'].iloc[0])
    original_results = baseline_result.copy()
    print(f'Baseline PPL re-evaluated: {baseline_ppl:.6f}')

experiments = [
    {
        'name': 'rmft_216',
        'method': 'rmft',
        'params': {'rmft_probability': 0.2161},
    },
    {
        'name': 'ner_strict',
        'method': 'tmft_ner',
        'params': {'ner_labels': ['PERSON', 'EMAIL', 'PHONE']},
    },
    {
        'name': 'ner_moderate',
        'method': 'tmft_ner',
        'params': {'ner_labels': ['PERSON', 'EMAIL', 'PHONE', 'GPE', 'LOC']},
    },
    {
        'name': 'mia_p90',
        'method': 'tmft_mia',
        'params': {'threshold_percentile': 90},
    },
    {
        'name': 'combined_p90',
        'method': 'tmft_combined',
        'params': {'threshold_percentile': 90, 'max_mask_ratio': 0.50},
    },
    {
        'name': 'combined_p90_skip65',
        'method': 'tmft_combined',
        'params': {'threshold_percentile': 90, 'max_mask_ratio': 0.65},
    },
]

ablation_results = []

for experiment in experiments:
    name = experiment['name']
    method = experiment['method']
    cfg = deepcopy(final_config)
    cfg.update(experiment['params'])
    cfg['seed'] = 42
    cfg['text_column'] = 'text'
    cfg['output_dir'] = f'./results/ablations/{name}'
    cfg['results_table_dir'] = f'./results/ablations/{name}/tables'

    expected_model_dir = Path(cfg['output_dir']) / method
    print(f'\n===== {name} =====')

    if (expected_model_dir / 'adapter_config.json').exists():
        model_dir = expected_model_dir
        print(f'Reusing existing adapter: {model_dir}')
    else:
        trainer, tokenizer, model_dir = train_model(
            cfg,
            method=method,
            train_dataset=final_splits['train'],
            eval_dataset=final_splits['validation'],
        )
        # Free the training model before loading a fresh evaluation model.
        del trainer, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    result = run_eval(
        cfg,
        method=method,
        splits=final_splits,
        eval_path=eval_path,
        model_dir=str(model_dir),
    )
    result['experiment'] = name
    result['mdp'] = result['ppl'] - baseline_ppl
    ablation_results.append(result)

ablation_df = pd.concat(ablation_results, ignore_index=True)

output_dir = Path('results/tables')
output_dir.mkdir(parents=True, exist_ok=True)
ablation_path = output_dir / 'ablation_results.csv'
ablation_df.to_csv(ablation_path, index=False)

# Also save a single comparison table containing the original five conditions.
original_comparison = original_results.copy()
original_comparison['experiment'] = original_comparison['method'].astype(str) + '_original'
expanded_results = pd.concat([original_comparison, ablation_df], ignore_index=True, sort=False)
expanded_path = output_dir / 'expanded_results.csv'
expanded_results.to_csv(expanded_path, index=False)

print(f'Baseline PPL used for every MDP: {baseline_ppl:.6f}')
print(f'Saved: {ablation_path}')
print(f'Saved: {expanded_path}')
ablation_df

## 10. Hugging Face Login

Use a token with **Write** permission. The token is entered through hidden input and is not written into the notebook.

In [ ]:
# Hugging Face login (prompt, hidden input)
import os
from getpass import getpass

try:
    from huggingface_hub import login
except Exception:
    %pip install -U huggingface_hub
    from huggingface_hub import login

print('Create token: https://huggingface.co/settings/tokens')

env_token = (os.environ.get('HF_TOKEN') or '').strip()
if env_token:
    use_env = input('HF_TOKEN is set. Use it? [Y/n]: ').strip().lower()
    token = env_token if use_env in ('', 'y', 'yes') else getpass('Enter HF token: ').strip()
else:
    token = getpass('Enter HF token: ').strip()

if not token:
    raise ValueError('No token provided.')

login(token=token)
os.environ['HF_TOKEN'] = token
print('Hugging Face login OK.')

## 11. Upload Model, Code, and Results

The validated `tmft_combined` LoRA adapter is uploaded at the model-repository root. Source code, configuration, notebook, tables, figures, and the comparison adapters are uploaded under subdirectories. Raw/processed Enron data and `data/pii_eval.json` are deliberately excluded.

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path

HF_REPO_ID = input('Repository ID (username/repository): ').strip()
PRIVATE_REPO = True  # Change to False only after reviewing every uploaded artifact.

if '/' not in HF_REPO_ID:
    raise ValueError('Use the form username/repository')

project_root = Path.cwd()
trained_paths = globals().get('train_outputs') or {
    method: str(project_root / 'results' / method)
    for method in ['baseline', 'rmft', 'tmft_ner', 'tmft_mia', 'tmft_combined']
}
primary_model_dir = Path(trained_paths.get('tmft_combined', 'results/tmft_combined'))
if not (primary_model_dir / 'adapter_config.json').exists():
    raise FileNotFoundError(f'LoRA adapter not found: {primary_model_dir}')

api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type='model',
    private=PRIVATE_REPO,
    exist_ok=True,
)

# Root remains directly loadable as a PEFT model repository.
api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type='model',
    folder_path=str(primary_model_dir),
    path_in_repo='',
    ignore_patterns=['checkpoint-*', '*.pt', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth'],
)

readme_path = project_root / 'README.md'
if readme_path.exists():
    api.upload_file(
        repo_id=HF_REPO_ID, repo_type='model', path_or_fileobj=str(readme_path), path_in_repo='README.md'
    )

for filename in ['requirements.txt', 'main.py', 'tmft_experiment.ipynb']:
    file_path = project_root / filename
    if file_path.exists():
        api.upload_file(
            repo_id=HF_REPO_ID,
            repo_type='model',
            path_or_fileobj=str(file_path),
            path_in_repo=f'project/{filename}',
        )

for local_dir, repo_dir in [
    ('src', 'project/src'),
    ('configs', 'project/configs'),
    ('results/tables', 'experiment/tables'),
    ('results/figures', 'experiment/figures'),
]:
    folder = project_root / local_dir
    if folder.exists():
        api.upload_folder(
            repo_id=HF_REPO_ID,
            repo_type='model',
            folder_path=str(folder),
            path_in_repo=repo_dir,
            ignore_patterns=['__pycache__', '*.pyc', '.ipynb_checkpoints'],
        )

# Small LoRA comparison adapters make the five-condition experiment reproducible.
for method, model_path in trained_paths.items():
    if method == 'tmft_combined':
        continue
    model_path = Path(model_path)
    if (model_path / 'adapter_config.json').exists():
        api.upload_folder(
            repo_id=HF_REPO_ID,
            repo_type='model',
            folder_path=str(model_path),
            path_in_repo=f'experiment/adapters/{method}',
            ignore_patterns=['checkpoint-*', '*.pt', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth'],
        )

print(f'Upload complete: https://huggingface.co/{HF_REPO_ID}')
print('Raw data and PII evaluation records were not uploaded.')

## 12. Download the Complete Vessel Project

This creates a ZIP backup of the current project directory, including data and results. Treat the archive as sensitive because it can contain Enron PII. The ZIP is created in `/tmp` first to avoid including itself recursively, then moved into the project directory for browser download.

In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

project_root = Path.cwd().resolve()
archive_destination = project_root / f'{project_root.name}_complete.zip'
archive_destination.unlink(missing_ok=True)

temporary_base = Path('/tmp') / f'{project_root.name}_complete'
temporary_zip = Path(shutil.make_archive(
    str(temporary_base),
    'zip',
    root_dir=str(project_root.parent),
    base_dir=project_root.name,
))
shutil.move(str(temporary_zip), str(archive_destination))

size_gb = archive_destination.stat().st_size / (1024 ** 3)
print(f'Archive ready: {archive_destination} ({size_gb:.2f} GB)')
print('Keep this local backup private; it may contain PII.')
display(FileLink(archive_destination.name, result_html_prefix='Download complete project: '))